In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()





# XGBoost

We'll use in this notebook an external library


### Requeriments


Make sure to use a ML Cluster, for instance 12.2 LTS ML






## Load Data & Preparation

Let's load the clean Airbnb dataset in again , but this time we will split in 3, so we have a validation set as well as a training and test sets

We created it in a previous notebook, it should exists in `dbfs:/FileStore/output/airbnb/clean_data` 

Also, let's index all of our categorical features, and set our label to be **`log(price)`**.


In [5]:
from pyspark.sql.functions import log, col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator

file_path = "/home/jovyan/work/datasets/outpus/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)

#split in train, val and test
train_df, val_df, test_df = airbnb_df.filter("price < 250").withColumn("label", log(col("price"))).randomSplit([.6, .2, .2], seed=42)

#Extract, prepare and index Categorical Columns
categorical_cols = [field for (field, dataType) in train_df.dtypes if dataType == "string"]
index_output_cols = [x + "Index" for x in categorical_cols]
string_indexer = StringIndexer(inputCols=categorical_cols, outputCols=index_output_cols, handleInvalid="skip")

#Extract numeric columns except the label and the price
numeric_cols = [field for (field, dataType) in train_df.dtypes if ((dataType == "double") & (field != "price") & (field != "label"))]

#Join all columns together and vector assemble them
assembler_inputs = index_output_cols + numeric_cols
vec_assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

regression_evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="price", metricName="rmse")


### Distributed Training of XGBoost Models

To create our distributed XGBoost model. We will use `xgboost`'s PySpark estimator. 

We need to specify two additional parameters:

* `num_workers`: The number of workers to distribute over.
* `use_gpu`: Enable to utilize GPU based training for faster performance.



In [6]:
from xgboost.spark import SparkXGBRegressor
from pyspark.ml import Pipeline

base_params = {"n_estimators": 100, "learning_rate": 0.1, "max_depth": 4, "random_state": 42, "missing": 0}



# Hyperopt



In [7]:
from hyperopt import hp
from hyperopt import fmin, tpe, Trials
import numpy as np
from pyspark.sql.functions import exp
from pyspark.ml.evaluation import RegressionEvaluator


def objective_function(params):    
    
    # set the hyperparameters that we want to tune
    max_depth = int(params["max_depth"])
    n_estimators = int(params["n_estimators"])
    learning_rate = int(params["learning_rate"])


    updated_params = base_params.copy()
    updated_params.update({'max_depth': max_depth, 'n_estimators': n_estimators, 'learning_rate': learning_rate})
    

    xgboost = SparkXGBRegressor(**updated_params)
    estimator = Pipeline(stages=[string_indexer, vec_assembler, xgboost])

    model = estimator.fit(train_df)

    preds = model.transform(val_df).withColumn("prediction", exp(col("prediction")))
    rmse = RegressionEvaluator(predictionCol="prediction", labelCol="price", metricName="rmse").evaluate(preds)

    return rmse


In [8]:
search_space = {
    "n_estimators": hp.quniform("n_estimators", 10, 200, 1),
    "max_depth": hp.quniform("max_depth", 2, 10, 1),
    "learning_rate": hp.uniform("learning_rate", 0.01, 0.3)  
}


In [9]:
num_evals = 5
trials = Trials()
best_hyperparam = fmin(fn=objective_function, 
                       space=search_space,
                       algo=tpe.suggest, 
                       max_evals=num_evals,
                       trials=trials,
                       rstate=np.random.default_rng(42))


  0%|          | 0/5 [00:00<?, ?trial/s, best loss=?]

2026-03-22 15:49:53,877 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'learning_rate': 0, 'max_depth': 7, 'random_state': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 165}
	dmatrix_kwargs: {'nthread': 1, 'missing': 0.0}

2026-03-22 15:49:58,773 INFO XGBoost-PySpark: _fit Finished xgboost training!



 20%|██        | 1/5 [00:07<00:28,  7.23s/trial, best loss: 59.251905440660195]

2026-03-22 15:49:59,673 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'learning_rate': 0, 'max_depth': 7, 'random_state': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 67}
	dmatrix_kwargs: {'nthread': 1, 'missing': 0.0}

2026-03-22 15:50:03,149 INFO XGBoost-PySpark: _fit Finished xgboost training!



 40%|████      | 2/5 [00:11<00:16,  5.43s/trial, best loss: 59.251905440660195]

...

 60%|██████    | 3/5 [00:15<00:09,  4.72s/trial, best loss: 59.251905440660195]

2026-03-22 15:50:07,614 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'learning_rate': 0, 'max_depth': 4, 'random_state': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 87}
	dmatrix_kwargs: {'nthread': 1, 'missing': 0.0}

2026-03-22 15:50:10,938 INFO XGBoost-PySpark: _fit Finished xgboost training!



 80%|████████  | 4/5 [00:19<00:04,  4.37s/trial, best loss: 59.251905440660195]

2026-03-22 15:50:11,432 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'learning_rate': 0, 'max_depth': 3, 'random_state': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 106}
	dmatrix_kwargs: {'nthread': 1, 'missing': 0.0}

2026-03-22 15:50:14,737 INFO XGBoost-PySpark: _fit Finished xgboost training!



100%|██████████| 5/5 [00:22<00:00,  4.58s/trial, best loss: 59.251905440660195]


In [10]:
print(best_hyperparam)


best_params = base_params.copy()

best_params.update({'max_depth': int(best_hyperparam["max_depth"]),
                        'n_estimators': int(best_hyperparam["n_estimators"]),
                         'learning_rate': best_hyperparam["learning_rate"]})

#Create Regressor
xgboost = SparkXGBRegressor(**best_params)

#Create pipeline
pipeline = Pipeline(stages=[string_indexer, vec_assembler, xgboost])

#Combine train and val_df
combined_df = train_df.union(val_df) # Combine train & validation together


#Train model
pipeline_model = pipeline.fit(combined_df)


{'learning_rate': 0.2663387213870585, 'max_depth': 7.0, 'n_estimators': 165.0}


2026-03-22 15:50:15,353 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'learning_rate': 0.2663387213870585, 'max_depth': 7, 'random_state': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 165}
	dmatrix_kwargs: {'nthread': 1, 'missing': 0.0}

2026-03-22 15:50:19,201 INFO XGBoost-PySpark: _fit Finished xgboost training!



In [11]:
pred_df = pipeline_model.transform(test_df).withColumn("prediction", exp(col("prediction")))

rmse = RegressionEvaluator(predictionCol="prediction", labelCol="price", metricName="rmse").evaluate(pred_df)
r2 = RegressionEvaluator(predictionCol="prediction", labelCol="price", metricName="r2").evaluate(pred_df)

print(f"RMSE is {rmse}")
print(f"R2 is {r2}")


RMSE is 35.603072691313564
R2 is 0.6198969725349823
